# Rollout Fine-tuning for 3D Temperature Surrogate



In [ ]:
from pathlib import Path
import json
import random
import sys
import gc

import numpy as np
import matplotlib.pyplot as plt
import torch
from IPython.display import clear_output, display

# Locate this rollout folder whether the notebook is opened from
# temperature_field_bundle/ or from rollout_surrogate/.
cwd = Path.cwd().resolve()
if (cwd / 'train_rollout_temperature_surrogate.py').exists():
    ROLLOUT_DIR = cwd
elif (cwd / 'rollout_surrogate' / 'train_rollout_temperature_surrogate.py').exists():
    ROLLOUT_DIR = cwd / 'rollout_surrogate'
else:
    raise FileNotFoundError('Open this notebook from temperature_field_bundle or rollout_surrogate.')

if str(ROLLOUT_DIR) not in sys.path:
    sys.path.insert(0, str(ROLLOUT_DIR))

import train_rollout_temperature_surrogate as r

print('Rollout dir:', ROLLOUT_DIR)
print('Bundle root:', r.BUNDLE_ROOT)
print('Run dir:', r.RUN_DIR)


## Training Settings



In [ ]:
# You can edit these before running the setup cell.
ROLLOUT_STEPS = r.ROLLOUT_STEPS
PATCH_SIZE = r.PATCH_SIZE
PATCH_SIZES = ((8, 32, 32), (16, 32, 32), (24, 16, 16))
EPOCHS = 20
STEPS_PER_EPOCH = 128
VAL_ROLLOUTS = 32
LONG_VAL_ROLLOUTS = 8
LONG_ROLLOUT_STEPS = 30
LEARNING_RATE = 1e-5
SEED = r.SEED

print('Rollout steps:', ROLLOUT_STEPS)
print('Default inference patch size:', PATCH_SIZE)
print('Training patch sizes:', PATCH_SIZES)
print('Z crop:', r.Z_CROP)
print('Epochs:', EPOCHS)
print('Steps per epoch:', STEPS_PER_EPOCH)
print('Validation rollouts:', VAL_ROLLOUTS)
print('Long validation rollouts:', LONG_VAL_ROLLOUTS)
print('Long validation rollout steps:', LONG_ROLLOUT_STEPS)
print('Learning rate:', LEARNING_RATE)


## Load Data and One-step Checkpoint


In [ ]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    torch.backends.cudnn.benchmark = True
    if hasattr(torch, 'set_float32_matmul_precision'):
        torch.set_float32_matmul_precision('high')

case_dirs = r.discover_case_dirs(r.DATA_ROOT)
cases = r.load_cases(case_dirs)
train_cases, val_cases = r.split_cases(cases, val_fraction=r.VAL_FRACTION, seed=SEED)
if not train_cases:
    train_cases = val_cases

if not r.ONE_STEP_CHECKPOINT.exists():
    raise FileNotFoundError(f'Missing one-step checkpoint: {r.ONE_STEP_CHECKPOINT}')
checkpoint = torch.load(r.ONE_STEP_CHECKPOINT, map_location='cpu')
stats = r.load_stats(checkpoint, train_cases)
ckpt_config = checkpoint.get('config', {})
in_channels = r.input_channel_count(r.HISTORY_STEPS)

model = r.UNet3D(
    in_channels=in_channels,
    out_channels=1,
    base_channels=int(ckpt_config.get('base_channels', r.BASE_CHANNELS)),
    depth=int(ckpt_config.get('depth', r.DEPTH)),
).to(device)
model.load_state_dict(r.adapt_state_dict_for_velocity_window(checkpoint['model_state_dict'], model))

helper = r.TemperatureSurrogateDataset(
    train_cases,
    stats,
    history_steps=r.HISTORY_STEPS,
    patch_size=PATCH_SIZE,
    samples_per_case=1,
    random_crop=True,
    deterministic_sampling=False,
    seed=SEED,
)
val_helper = r.TemperatureSurrogateDataset(
    val_cases,
    stats,
    history_steps=r.HISTORY_STEPS,
    patch_size=PATCH_SIZE,
    samples_per_case=1,
    random_crop=True,
    deterministic_sampling=True,
    seed=SEED + 10_000,
)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=r.WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
scaler = torch.amp.GradScaler('cuda', enabled=r.AMP and device.type == 'cuda')
train_rng = random.Random(SEED + 1)
fixed_val_samples = r.make_fixed_validation_samples(
    val_cases,
    count=VAL_ROLLOUTS,
    seed=SEED + 20_000,
    patch_size=None,
    patch_sizes=PATCH_SIZES,
    rollout_steps=ROLLOUT_STEPS,
)
fixed_long_val_samples = r.make_fixed_validation_samples(
    val_cases,
    count=LONG_VAL_ROLLOUTS,
    seed=SEED + 30_000,
    patch_size=None,
    patch_sizes=PATCH_SIZES,
    rollout_steps=LONG_ROLLOUT_STEPS,
)

late_step_loss_gamma = getattr(r, 'LATE_STEP_LOSS_GAMMA', 1.0)
use_boundary_temperature_constraints = getattr(r, 'USE_BOUNDARY_TEMPERATURE_CONSTRAINTS', False)

run_config = {
    'mode': 'rollout_finetune_notebook_lr1e-5_epoch20',
    'created_from_checkpoint': str(r.ONE_STEP_CHECKPOINT),
    'history_steps': r.HISTORY_STEPS,
    'rollout_steps': ROLLOUT_STEPS,
    'long_rollout_steps': LONG_ROLLOUT_STEPS,
    'z_crop': r.Z_CROP,
    'patch_size': PATCH_SIZE,
    'patch_sizes': [list(patch_size) for patch_size in PATCH_SIZES],
    'steps_per_epoch': STEPS_PER_EPOCH,
    'epochs': EPOCHS,
    'val_rollouts': VAL_ROLLOUTS,
    'long_val_rollouts': LONG_VAL_ROLLOUTS,
    'learning_rate': LEARNING_RATE,
    'loss': {
        'base': 'masked_mse',
        'gradient_loss_weight': r.GRADIENT_LOSS_WEIGHT,
        'multiscale_loss_weight': r.MULTISCALE_LOSS_WEIGHT,
        'multiscale_factors': list(r.MULTISCALE_FACTORS),
        'vertical_profile_loss_weight': r.VERTICAL_PROFILE_LOSS_WEIGHT,
        'late_step_loss_gamma': late_step_loss_gamma,
    },
    'use_next_velocity': r.USE_NEXT_VELOCITY,
    'use_implicit_future_guess': r.USE_IMPLICIT_FUTURE_GUESS,
    'future_guess_mode': r.FUTURE_GUESS_MODE,
    'future_guess_clip_c': r.FUTURE_GUESS_CLIP_C,
    'correction_iterations': r.CORRECTION_ITERATIONS,
    'use_cool_surface_channels': r.USE_COOL_SURFACE_CHANNELS,
    'use_cool_surface_relaxation': r.USE_COOL_SURFACE_RELAXATION,
    'use_boundary_temperature_constraints': use_boundary_temperature_constraints,
    'cool_surface_relaxation': r.COOL_SURFACE_RELAXATION,
    'open_ground_relaxation': r.OPEN_GROUND_RELAXATION,
    'cool_surface_vertical_decay_layers': r.COOL_SURFACE_VERTICAL_DECAY_LAYERS,
    'cool_surface_oversampling': False,
    'state_dict_adaptation': 'zero_init_new_channels_preserve_onestep',
    'in_channels': in_channels,
    'base_channels': int(ckpt_config.get('base_channels', r.BASE_CHANNELS)),
    'depth': int(ckpt_config.get('depth', r.DEPTH)),
    'train_cases': [str(case.case_dir) for case in train_cases],
    'val_cases': [str(case.case_dir) for case in val_cases],
    'fixed_validation_samples_path': str(r.RUN_DIR / 'fixed_validation_samples.json'),
    'fixed_long_validation_samples_path': str(r.RUN_DIR / 'fixed_long_validation_samples.json'),
}
r.RUN_DIR.mkdir(parents=True, exist_ok=True)
(r.RUN_DIR / 'fixed_validation_samples.json').write_text(json.dumps(fixed_val_samples, indent=2))
(r.RUN_DIR / 'fixed_long_validation_samples.json').write_text(json.dumps(fixed_long_val_samples, indent=2))
(r.RUN_DIR / 'rollout_training_config.json').write_text(json.dumps(run_config, indent=2))

print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
    print('Total GPU memory GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print(f'Cases: {len(cases)} | train: {len(train_cases)} | val: {len(val_cases)}')
print('Loaded one-step checkpoint:', r.ONE_STEP_CHECKPOINT)
print('Output dir:', r.RUN_DIR)
print('Fixed validation samples:', len(fixed_val_samples))
print('Fixed long validation samples:', len(fixed_long_val_samples), 'steps:', LONG_ROLLOUT_STEPS)
print('Loss: masked MSE +', r.GRADIENT_LOSS_WEIGHT, '* gradient +', r.MULTISCALE_LOSS_WEIGHT, '* multiscale')
print('Use velocity[t+1]:', r.USE_NEXT_VELOCITY)
print('Correction iterations per time step:', r.CORRECTION_ITERATIONS)
print('Use cool-surface channels:', r.USE_COOL_SURFACE_CHANNELS)
print('Use cool-surface relaxation:', r.USE_COOL_SURFACE_RELAXATION)
print('Cool-surface relaxation:', r.COOL_SURFACE_RELAXATION, 'vegetation |', r.OPEN_GROUND_RELAXATION, 'open ground')
print('Vertical-profile loss weight:', r.VERTICAL_PROFILE_LOSS_WEIGHT)
print('Late-step loss gamma:', late_step_loss_gamma)
print('Use boundary temperature constraints:', use_boundary_temperature_constraints)
print('Cool-surface oversampling: False')
print('Input channels:', in_channels)


## Rollout Fine-tuning



In [ ]:
history = {
    'train_loss': [],
    'val_loss': [],
    'long_val_loss': [],
    'train_mae_c': [],
    'val_mae_c': [],
    'long_val_mae_c': [],
    'val_bias_c': [],
    'long_val_bias_c': [],
}
best_val = float('inf')
best_path = r.RUN_DIR / 'best_temperature_3d_unet_surrogate_rollout.pt'
last_path = r.RUN_DIR / 'last_temperature_3d_unet_surrogate_rollout.pt'

def plot_history(history):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    epochs = range(1, len(history['train_loss']) + 1)
    axes[0].plot(epochs, history['train_loss'], label='train')
    axes[0].plot(epochs, history['val_loss'], label=f'val {ROLLOUT_STEPS}-step')
    axes[0].plot(epochs, history['long_val_loss'], label=f'val {LONG_ROLLOUT_STEPS}-step')
    axes[0].set_title(f'{ROLLOUT_STEPS}-step training loss')
    axes[0].set_xlabel('epoch')
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()
    axes[1].plot(epochs, history['train_mae_c'], label='train')
    axes[1].plot(epochs, history['val_mae_c'], label=f'val {ROLLOUT_STEPS}-step')
    axes[1].plot(epochs, history['long_val_mae_c'], label=f'val {LONG_ROLLOUT_STEPS}-step')
    axes[1].set_title('Rollout MAE (deg C)')
    axes[1].set_xlabel('epoch')
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()
    axes[2].plot(epochs, history['val_bias_c'], label=f'val {ROLLOUT_STEPS}-step')
    axes[2].plot(epochs, history['long_val_bias_c'], label=f'val {LONG_ROLLOUT_STEPS}-step')
    axes[2].axhline(0.0, color='black', linewidth=1)
    axes[2].set_title('Rollout signed bias')
    axes[2].set_xlabel('epoch')
    axes[2].set_ylabel('deg C')
    axes[2].grid(True, alpha=0.3)
    axes[2].legend()
    plt.tight_layout()
    display(fig)
    plt.close(fig)

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_losses = []
    train_maes = []
    for step in range(1, STEPS_PER_EPOCH + 1):
        case = train_rng.choice(train_cases)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=r.AMP and device.type == 'cuda'):
            loss, mae, bias = r.sample_rollout_loss(model, case, stats, helper, device, train_rng)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), r.GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()
        train_losses.append(float(loss.detach().cpu()))
        train_maes.append(float(mae.detach().cpu()))
        if step % 64 == 0:
            print(f'Epoch {epoch}/{EPOCHS} step {step}/{STEPS_PER_EPOCH}: loss={np.mean(train_losses[-64:]):.5f}, MAE={np.mean(train_maes[-64:]):.3f} C', flush=True)
        del loss, mae, bias

    val_metrics = r.evaluate_fixed_rollout(
        model,
        val_cases,
        stats,
        val_helper,
        device,
        fixed_val_samples,
        patch_size=None,
        rollout_steps=ROLLOUT_STEPS,
    )
    long_val_metrics = r.evaluate_fixed_rollout(
        model,
        val_cases,
        stats,
        val_helper,
        device,
        fixed_long_val_samples,
        patch_size=None,
        rollout_steps=LONG_ROLLOUT_STEPS,
    )
    train_loss = float(np.mean(train_losses))
    train_mae = float(np.mean(train_maes))
    scheduler.step(val_metrics['loss'])

    history['train_loss'].append(train_loss)
    history['train_mae_c'].append(train_mae)
    history['val_loss'].append(val_metrics['loss'])
    history['val_mae_c'].append(val_metrics['mae_c'])
    history['long_val_loss'].append(long_val_metrics['loss'])
    history['long_val_mae_c'].append(long_val_metrics['mae_c'])
    history['val_bias_c'].append(val_metrics['bias_c'])
    history['long_val_bias_c'].append(long_val_metrics['bias_c'])
    (r.RUN_DIR / 'rollout_training_history.json').write_text(json.dumps(history, indent=2))

    payload = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'stats': stats.__dict__,
        'config': run_config,
        'history': history,
        'epoch': epoch,
        'best_val_loss': min(best_val, val_metrics['loss']),
        'long_val_loss': long_val_metrics['loss'],
        'long_val_mae_c': long_val_metrics['mae_c'],
        'long_val_bias_c': long_val_metrics['bias_c'],
    }
    torch.save(payload, last_path)
    is_best = val_metrics['loss'] < best_val
    if is_best:
        best_val = val_metrics['loss']
        payload['best_val_loss'] = best_val
        torch.save(payload, best_path)

    clear_output(wait=True)
    print(f'Epoch {epoch}/{EPOCHS}')
    print(f"train loss={train_loss:.5f}, val loss={val_metrics['loss']:.5f}, long val loss={long_val_metrics['loss']:.5f}")
    print(f"train rollout MAE={train_mae:.3f} C, val rollout MAE={val_metrics['mae_c']:.3f} C, long val MAE={long_val_metrics['mae_c']:.3f} C")
    print(f"val bias={val_metrics['bias_c']:+.3f} C, long val bias={long_val_metrics['bias_c']:+.3f} C")
    print('best checkpoint:', best_path)
    print('last checkpoint:', last_path)
    print('saved best this epoch:', is_best)
    plot_history(history)

    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()

print('Saved history to', r.RUN_DIR / 'rollout_training_history.json')
